# 14 — P2: MILP PWL Primitives

**Plan 2 — Phase 4.1.** Sound piecewise-linear (PWL) lower- and upper-bracketing
envelopes for the four scalar nonlinearities used in the ViT MILP encoder.
Every function is encoded with a *bracket*: a lower PWL `f_lo(x) ≤ f(x)` and an
upper PWL `f_up(x) ≥ f(x)`, both piecewise-linear in `x` over a known interval
`[a, b]`. The MILP variable representing `f(x)` is constrained to lie in
`[f_lo(x), f_up(x)]`, giving sound (over-approximating) bounds.

## Functions and convexity
| `f` | domain | convexity | lower envelope | upper envelope |
|---|---|---|---|---|
| `exp(x)` | from IBP on attention scores | convex | tangents at midpoints | secants between breakpoints |
| `1/x`, `x>0` | softmax denominator | convex | tangents | secants |
| `sqrt(x)`, `x>0` | RMSNorm | concave | secants | tangents |
| `x²` | RMSNorm input squaring | convex | tangents | secants |

## Big-M disjunctive encoding (per segment k of n)
```
Σ_k δ_k = 1                                          # exactly one segment active
b_k − M·(1−δ_k)   ≤ x    ≤ b_{k+1} + M·(1−δ_k)        # segment selects x ∈ [b_k, b_{k+1}]
lo_k(x) − M·(1−δ_k) ≤ y ≤ up_k(x) + M·(1−δ_k)         # bracket on the active segment
```
where `lo_k(x) = slope_lo_k·x + int_lo_k` and `up_k(x) = slope_up_k·x + int_up_k`.

## Tests in this notebook
For each function `f`:
1. Construct `n_pieces=8` PWL bracket on a representative domain.
2. Sample `K` inputs uniformly in the domain; verify analytically that
   `f_lo_pwl(x) ≤ f(x) ≤ f_up_pwl(x)` at every sample.
3. Build a small Gurobi model that encodes the bracket; for each sampled `x`,
   pin the input variable and check that the MILP `y` variable's feasible
   interval brackets `f(x)`.
4. Report max absolute error of upper minus true and true minus lower.

In [1]:
!pip install -q numpy gurobipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 89.4 MB/s eta 0:00:00:00:010:01


In [ ]:
from __future__ import annotations
import math, json
from dataclasses import dataclass
from typing import Callable, List, Tuple
import numpy as np

try:
    import gurobipy as gp
    from gurobipy import GRB
    HAS_GUROBI = True
except Exception as e:
    HAS_GUROBI = False
    print(f'Gurobi not available ({e}); will skip MILP-side tests.')

rng = np.random.default_rng(1234)

In [ ]:
# ── PWL bracket data structure ────────────────────────────────────────────
@dataclass
class PWLBracket:
    """
    Sound piecewise-linear bracket for a scalar function f on [a, b].

    breakpoints : list of n+1 floats, strictly increasing
    slope_lo    : list of n floats — lower envelope slope per segment
    int_lo      : list of n floats — lower envelope intercept per segment
    slope_up    : list of n floats — upper envelope slope per segment
    int_up      : list of n floats — upper envelope intercept per segment

    On segment k = [breakpoints[k], breakpoints[k+1]]:
        slope_lo[k]·x + int_lo[k]  ≤  f(x)  ≤  slope_up[k]·x + int_up[k]
    """
    name: str
    breakpoints: List[float]
    slope_lo: List[float]
    int_lo:   List[float]
    slope_up: List[float]
    int_up:   List[float]

    @property
    def n_pieces(self) -> int:
        return len(self.breakpoints) - 1

    @property
    def domain(self) -> Tuple[float, float]:
        return (self.breakpoints[0], self.breakpoints[-1])

    def evaluate(self, x: float) -> Tuple[float, float]:
        """Return (lo, up) PWL values at x.  x must lie in the domain."""
        a, b = self.domain
        x_c = max(a, min(b, x))
        # binary-search-free: scan (n_pieces is small, ≤16)
        for k in range(self.n_pieces):
            if self.breakpoints[k] - 1e-12 <= x_c <= self.breakpoints[k+1] + 1e-12:
                lo = self.slope_lo[k] * x_c + self.int_lo[k]
                up = self.slope_up[k] * x_c + self.int_up[k]
                return float(lo), float(up)
        raise RuntimeError(f'x={x} not in segments of {self.name}')

In [4]:
# ── Bracket builders ──────────────────────────────────────────────────────
def _line_through(p1: Tuple[float, float], p2: Tuple[float, float]) -> Tuple[float, float]:
    x1, y1 = p1; x2, y2 = p2
    slope = (y2 - y1) / (x2 - x1)
    intc  = y1 - slope * x1
    return slope, intc

def _tangent_at(f: Callable[[float], float], df: Callable[[float], float],
                x0: float) -> Tuple[float, float]:
    s = df(x0); intc = f(x0) - s * x0
    return s, intc

def build_pwl_convex(f, df, a: float, b: float, n_pieces: int, name: str) -> PWLBracket:
    """
    f convex on [a, b]:
        upper envelope = secants between consecutive breakpoints
        lower envelope = tangents at segment midpoints
    """
    bps = list(np.linspace(a, b, n_pieces + 1))
    slope_up, int_up, slope_lo, int_lo = [], [], [], []
    for k in range(n_pieces):
        x_l, x_r = bps[k], bps[k+1]
        # secant (upper for convex)
        s_u, i_u = _line_through((x_l, f(x_l)), (x_r, f(x_r)))
        slope_up.append(s_u); int_up.append(i_u)
        # tangent at midpoint (lower for convex)
        x_m = 0.5 * (x_l + x_r)
        s_l, i_l = _tangent_at(f, df, x_m)
        slope_lo.append(s_l); int_lo.append(i_l)
    return PWLBracket(name=name, breakpoints=bps,
                      slope_lo=slope_lo, int_lo=int_lo,
                      slope_up=slope_up, int_up=int_up)

def build_pwl_concave(f, df, a: float, b: float, n_pieces: int, name: str) -> PWLBracket:
    """
    f concave on [a, b]:
        upper envelope = tangents at segment midpoints
        lower envelope = secants between consecutive breakpoints
    """
    bps = list(np.linspace(a, b, n_pieces + 1))
    slope_up, int_up, slope_lo, int_lo = [], [], [], []
    for k in range(n_pieces):
        x_l, x_r = bps[k], bps[k+1]
        s_l, i_l = _line_through((x_l, f(x_l)), (x_r, f(x_r)))
        slope_lo.append(s_l); int_lo.append(i_l)
        x_m = 0.5 * (x_l + x_r)
        s_u, i_u = _tangent_at(f, df, x_m)
        slope_up.append(s_u); int_up.append(i_u)
    return PWLBracket(name=name, breakpoints=bps,
                      slope_lo=slope_lo, int_lo=int_lo,
                      slope_up=slope_up, int_up=int_up)

# ── The four primitives ───────────────────────────────────────────────────
def pwl_exp(a: float, b: float, n_pieces: int = 8) -> PWLBracket:
    return build_pwl_convex(math.exp, math.exp, a, b, n_pieces, name='exp')

def pwl_inv_pos(a: float, b: float, n_pieces: int = 8) -> PWLBracket:
    """1/x on [a, b] with a > 0 — convex."""
    assert a > 0, 'pwl_inv_pos requires a > 0'
    return build_pwl_convex(lambda x: 1.0/x, lambda x: -1.0/(x*x),
                            a, b, n_pieces, name='inv_pos')

def pwl_sqrt_pos(a: float, b: float, n_pieces: int = 8) -> PWLBracket:
    """sqrt(x) on [a, b] with a > 0 — concave."""
    assert a > 0, 'pwl_sqrt_pos requires a > 0'
    return build_pwl_concave(math.sqrt, lambda x: 0.5/math.sqrt(x),
                             a, b, n_pieces, name='sqrt_pos')

def pwl_square(a: float, b: float, n_pieces: int = 8) -> PWLBracket:
    """x² on [a, b] — convex."""
    return build_pwl_convex(lambda x: x*x, lambda x: 2*x, a, b, n_pieces, name='square')

In [ ]:
# ── Analytic soundness test ───────────────────────────────────────────────
def analytic_check(bracket: PWLBracket, f: Callable[[float], float],
                   n_samples: int = 500) -> dict:
    a, b = bracket.domain
    xs = np.linspace(a, b, n_samples)
    los, ups, trues = [], [], []
    for x in xs:
        lo, up = bracket.evaluate(float(x))
        true = f(float(x))
        assert lo <= true + 1e-9, f'{bracket.name}: lower violated at x={x}: lo={lo} > true={true}'
        assert true <= up + 1e-9, f'{bracket.name}: upper violated at x={x}: true={true} > up={up}'
        los.append(lo); ups.append(up); trues.append(true)
    los, ups, trues = np.array(los), np.array(ups), np.array(trues)
    return dict(
        max_under = float((trues - los).max()),  # how much lower envelope under-shoots
        max_over  = float((ups - trues).max()),  # how much upper envelope over-shoots
        max_gap   = float((ups - los).max()),
        mean_gap  = float((ups - los).mean()),
    )

tests = {
    'exp on [-3, 3]'        : (pwl_exp(-3.0, 3.0, n_pieces=8),    math.exp),
    'inv on [0.1, 10]'      : (pwl_inv_pos(0.1, 10.0, n_pieces=8),lambda x: 1.0/x),
    'sqrt on [0.01, 10]'    : (pwl_sqrt_pos(0.01, 10.0, n_pieces=8), math.sqrt),
    'square on [-2, 2]'     : (pwl_square(-2.0, 2.0, n_pieces=8), lambda x: x*x),
}
for name, (br, f) in tests.items():
    r = analytic_check(br, f)
    print(f'{name:25s}  max_gap={r["max_gap"]:.4e}  mean_gap={r["mean_gap"]:.4e}  '
          f'(under={r["max_under"]:.2e}, over={r["max_over"]:.2e})')

exp on [-3, 3]             max_gap=1.1042e+00  mean_gap=2.3278e-01  (under=1.10e+00, over=9.86e-01)
inv on [0.1, 10]           max_gap=7.4110e+00  mean_gap=5.1410e-01  (under=7.41e+00, over=5.28e+00)
sqrt on [0.01, 10]         max_gap=3.0452e-01  mean_gap=2.8944e-02  (under=2.14e-01, over=3.05e-01)
square on [-2, 2]          max_gap=6.2500e-02  mean_gap=6.2500e-02  (under=6.25e-02, over=6.25e-02)


In [ ]:
# ── Gurobi big-M encoder for a PWL bracket ────────────────────────────────
def add_pwl_bracket(model, x_var, y_var, bracket: PWLBracket,
                    big_M: float | None = None, prefix: str = ''):
    """
    Add the disjunctive big-M constraints encoding
        f_lo(x) ≤ y ≤ f_up(x)
    on the PWL bracket.  Returns the list of binary segment indicators δ_k.

    big_M defaults to a tight value derived from the bracket itself.
    """
    if not HAS_GUROBI:
        raise RuntimeError('Gurobi not available')
    n = bracket.n_pieces
    bps = bracket.breakpoints
    a, b = bps[0], bps[-1]

    if big_M is None:
        # Tight-ish big-M: max of |slope|·(b-a)+|int| over all segments, ×2
        scope = max(
            max(abs(bracket.slope_lo[k]) * (b - a) + abs(bracket.int_lo[k]) for k in range(n)),
            max(abs(bracket.slope_up[k]) * (b - a) + abs(bracket.int_up[k]) for k in range(n)),
        )
        big_M = max(2.0 * scope, 1.0)

    delta = [model.addVar(vtype=GRB.BINARY, name=f'{prefix}delta_{k}') for k in range(n)]
    model.addConstr(gp.quicksum(delta) == 1, name=f'{prefix}delta_sum')

    for k in range(n):
        x_l, x_r = bps[k], bps[k+1]
        # x in segment when δ_k = 1
        model.addConstr(x_var >= x_l - big_M * (1 - delta[k]), name=f'{prefix}seg{k}_xlo')
        model.addConstr(x_var <= x_r + big_M * (1 - delta[k]), name=f'{prefix}seg{k}_xup')
        # bracket on y when δ_k = 1
        s_lo, i_lo = bracket.slope_lo[k], bracket.int_lo[k]
        s_up, i_up = bracket.slope_up[k], bracket.int_up[k]
        model.addConstr(y_var >= s_lo * x_var + i_lo - big_M * (1 - delta[k]),
                        name=f'{prefix}seg{k}_ylo')
        model.addConstr(y_var <= s_up * x_var + i_up + big_M * (1 - delta[k]),
                        name=f'{prefix}seg{k}_yup')
    return delta

In [7]:
# ── End-to-end MILP test: solve for tightest y given pinned x ─────────────
#
# For each PWL bracket, sample K test inputs in the domain.
# Build a Gurobi model with a free x ∈ [a, b], free y, the bracket constraints,
# and pin x to the test value via x == x_pin.
# Solve twice: once minimising y (gives y_lo), once maximising y (gives y_up).
# Verify y_lo ≤ f(x_pin) ≤ y_up.

def milp_bracket_test(bracket: PWLBracket, f: Callable[[float], float],
                      K: int = 30, time_limit: float = 5.0) -> dict:
    if not HAS_GUROBI:
        return {'skipped': True}
    a, b = bracket.domain
    xs = np.linspace(a + 1e-3 * (b-a), b - 1e-3 * (b-a), K)

    max_lo_violation = 0.0
    max_up_violation = 0.0
    max_observed_gap = 0.0
    fail = []

    for x_pin in xs:
        m = gp.Model(f'pwl_test_{bracket.name}')
        m.setParam('OutputFlag', 0)
        m.setParam('TimeLimit', time_limit)
        x = m.addVar(lb=a, ub=b, vtype=GRB.CONTINUOUS, name='x')
        y = m.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY, vtype=GRB.CONTINUOUS, name='y')
        add_pwl_bracket(m, x, y, bracket)
        m.addConstr(x == float(x_pin), name='pin_x')

        # min y
        m.setObjective(y, GRB.MINIMIZE); m.optimize()
        if m.Status != GRB.OPTIMAL:
            fail.append((x_pin, 'min', m.Status)); continue
        y_lo = y.X
        # max y
        m.setObjective(y, GRB.MAXIMIZE); m.optimize()
        if m.Status != GRB.OPTIMAL:
            fail.append((x_pin, 'max', m.Status)); continue
        y_up = y.X

        true = f(float(x_pin))
        max_lo_violation = max(max_lo_violation, y_lo - true)
        max_up_violation = max(max_up_violation, true - y_up)
        max_observed_gap = max(max_observed_gap, y_up - y_lo)

    return dict(
        n_tested            = K,
        max_lower_violation = float(max_lo_violation),  # >0 means UNSOUND
        max_upper_violation = float(max_up_violation),  # >0 means UNSOUND
        max_observed_gap    = float(max_observed_gap),
        n_failed            = len(fail),
    )

if HAS_GUROBI:
    print('── MILP bracket tests (n_pieces=8, K=30 samples each) ─────────────')
    for name, (br, f) in tests.items():
        r = milp_bracket_test(br, f, K=30)
        sound = (r['max_lower_violation'] < 1e-6 and r['max_upper_violation'] < 1e-6)
        print(f'  {name:25s}  max_gap={r["max_observed_gap"]:.4e}  '
              f'lo_viol={r["max_lower_violation"]:.2e}  '
              f'up_viol={r["max_upper_violation"]:.2e}  '
              f'{"SOUND" if sound else "UNSOUND ✗"}')
else:
    print('Gurobi not installed — skipping MILP-side soundness tests.')

── MILP bracket tests (n_pieces=8, K=30 samples each) ─────────────
Restricted license - for non-production use only - expires 2027-11-29
  exp on [-3, 3]             max_gap=1.1023e+00  lo_viol=0.00e+00  up_viol=0.00e+00  SOUND
  inv on [0.1, 10]           max_gap=7.3561e+00  lo_viol=0.00e+00  up_viol=0.00e+00  SOUND
  sqrt on [0.01, 10]         max_gap=3.0261e-01  lo_viol=0.00e+00  up_viol=0.00e+00  SOUND
  square on [-2, 2]          max_gap=6.2500e-02  lo_viol=0.00e+00  up_viol=0.00e+00  SOUND


In [8]:
# ── Save bracket constructors as a module-like exportable artifact ────────
#
# Downstream notebooks (RMSNorm-MILP, attention-MILP, full-ViT-MILP) can
# re-define these helpers inline (the notebooks are self-contained per the
# repo convention), or load this JSON to inspect bracket tightness data.

from pathlib import Path
out_dir = Path('results/vit_p2'); out_dir.mkdir(parents=True, exist_ok=True)

summary = {}
for name, (br, f) in tests.items():
    r_a = analytic_check(br, f)
    r_m = milp_bracket_test(br, f, K=30) if HAS_GUROBI else {'skipped': True}
    summary[name] = dict(
        domain      = list(br.domain),
        n_pieces    = br.n_pieces,
        analytic    = r_a,
        milp        = r_m,
        breakpoints = list(map(float, br.breakpoints)),
    )

(out_dir / 'pwl_brackets.json').write_text(json.dumps(summary, indent=2))
print(f'Saved → {out_dir / "pwl_brackets.json"}')
print('\nPhase 4.1 complete: PWL primitives + analytic + MILP soundness verified.')
print('Next: Phase 4.2 (RMSNorm MILP encoding using these primitives).')

Saved → results/vit_p2/pwl_brackets.json

Phase 4.1 complete: PWL primitives + analytic + MILP soundness verified.
Next: Phase 4.2 (RMSNorm MILP encoding using these primitives).


In [9]:
from pathlib import Path

drive_out = Path('/content/drive/My Drive/thesis-formal-verification/results/vit_p2')
drive_out.mkdir(parents=True, exist_ok=True)
(drive_out / 'pwl_brackets.json').write_text((out_dir / 'pwl_brackets.json').read_text())
print(f'Saved drive copy → {drive_out / "pwl_brackets.json"}')

Saved drive copy → /content/drive/My Drive/thesis-formal-verification/results/vit_p2/pwl_brackets.json
